In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

RUN = Path("results/runs/latest") # symlink

solves = pd.read_csv(RUN / "solves.csv")
joints = pd.read_csv(RUN / "joints.csv")
constraints = pd.read_csv(RUN / "constraints.csv")
timings = pd.read_csv(RUN / "timings.csv")

print(f"{len(solves)} solve rows | solvers: {sorted(solves.solver_name.unique())}")
print(f"tols: {sorted(solves.orientation_tol.unique())} | seeds: {sorted(solves.seed.unique())}")
print(f"methods: {sorted(solves.generation_method.unique())}")
solves.head()

# feasibility

In [ ]:
(solves
 .groupby(["orientation_tol", "solver_name"])
 .agg(n=("success", "size"),
      success_rate=("success", "mean"),
      feasible_rate=("feasible", "mean"),
      nan_rate=("error_type", lambda s: (s == "NonFiniteSolution").mean()))
 .round(4))

In [ ]:
(solves
 .groupby(["solver_name"])
 .agg(n=("success", "size"),
      success_rate=("success", "mean"),
      feasible_rate=("feasible", "mean"),
      nan_rate=("error_type", lambda s: (s == "NonFiniteSolution").mean()))
 .round(3))

In [ ]:
fscens = solves[solves.feasible].scenario_id.unique()
n_scens = len(solves.scenario_id.unique())

feas_scens = solves[solves["scenario_id" ].isin(fscens)]

print(f"total scenarios: {n_scens}")
print(f"scenarios (>=1 feasible soln found): {len(fscens)} ({round(len(fscens)/n_scens, 4)*100}%)")
print(f"solve rows (only feasible scenarios): {len(feas_scens)}")

(feas_scens
 .groupby(["solver_name"])
 .agg(n=("success", "size"),
      success_rate=("success", "mean"),
      feasible_rate=("feasible", "mean"),
      nan_rate=("error_type", lambda s: (s == "NonFiniteSolution").mean()))
 .round(3))


# timing

In [ ]:
# median time - total by solver
(solves[solves.feasible]                      # only time solves that actually worked
 .groupby(["solver_name"])["time_median"]
 .median()
 .mul(1000).round(2)                          # ms
)

In [ ]:
# median solve time - by solver x orientation
(solves[solves.feasible]                      # only time solves that actually worked
 .groupby(["orientation_tol", "solver_name"])["time_median"]
 .median()
 .mul(1000).round(2)                          # ms
 .unstack("solver_name"))

In [ ]:
# pivot so each scenario has solvers side by side
piv = solves.pivot_table(
    index=["seed", "orientation_tol", "scenario_id"],
    columns="solver_name",
    values=["feasible", "common_stationarity_inf", "evaluated_cost"],
    aggfunc="first")

both_feas = piv[("feasible", "SNOPT")].astype(bool) & piv[("feasible", "Ipopt")].astype(bool) 
comp = piv[both_feas]
print(f"{both_feas.sum()} scenarios where both SNOPT and IPOPT feasible")

# stationarity residuals side by side, where both feasible
comp[[("common_stationarity_inf", "SNOPT"), ("common_stationarity_inf", "Ipopt")]].describe()

In [ ]:
cost_snopt = comp[("evaluated_cost", "SNOPT")]
cost_ipopt = comp[("evaluated_cost", "Ipopt")]

rel_gap = ((cost_snopt - cost_ipopt).abs() / cost_ipopt.abs())
abs_gap = cost_snopt - cost_ipopt
print("relative cost gap SNOPT vs IPOPT (both feasible):")
print(rel_gap.describe())